# Notebook 5: Deep feature alignment (Deep CORAL and JMMD)

These are the first two of our **research methods**. They belong to one family, so one notebook handles both. A switch in
Step 1 chooses which one to run; run the notebook **twice**, once for each.

### The shared idea
Notebook 4c (DANN) pulled the lab and field features together with an *adversary*. Here we do it with a plain
**penalty**. We train a small network (a *bottleneck* layer of 256 numbers, then a classifier) on lab images, and while
doing so we add a term to the loss that grows when the **lab features and the field features look different**:

**loss = classification loss (lab labels) + weight x alignment loss (lab features vs field features)**

The field images are the ~200 **calibration** images of each split, used **without labels**, exactly as for the baselines.

### The two alignment losses
* **Deep CORAL** (Sun & Saenko, 2016) compares the **covariance** of the two sets of features, meaning how the features vary
  *together*. It does **not** look at the averages. (Moment matching, our baseline, matched averages and spreads one feature
  at a time. CORAL looks at relationships between features.)
* **JMMD** (Long et al., 2017) compares the **whole distribution**, using a kernel trick called *maximum mean discrepancy*.
  "Joint" means it compares the bottleneck features **and the predicted class probabilities together**, which makes it
  partly class-aware without needing any field labels.

### Choosing the weight, without target labels
How strongly to align (the *weight*) matters a lot, and we have no field labels to tune it. We use a rule that needs **only
lab data**: try several weights on one development split and keep **the strongest weight that does not hurt the lab
classifier** (lab validation accuracy within 1 point of the no-alignment run) **and does not shrink the features**
(their average length must stay at least half of what it is without alignment). The second condition closes a loophole:
a covariance penalty can be made small simply by scaling the features down, which "aligns" nothing and only makes the
model less confident. This is our own choice, and the notebook shows the whole sweep so you can see it. Both methods get the **same** search budget.

### A matched control: `ALIGNMENT = "none"`
Both methods (and DANN in 4c) use a small bottleneck network, which is a *different* classifier from the fine-tuned ResNet
we call `source_only`. To know what the **alignment itself** adds, we also run the **same network with no alignment penalty**
(`ALIGNMENT = "none"`, saved as `bottleneck_only`). Compare each method with this control, not only with `source_only`.

### Plan
1. Settings
2. Load the features
3. The two alignment losses (with toy examples)
4. The model and the training function
5. Choose the weight
6. All 10 splits, then save
7. Evaluate with notebook 3b

## Step 1: Settings

In [14]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

In [15]:
# ---- Edit these -------------------------------------------------------------
ALIGNMENT = "jmmd"               # "coral", "jmmd", or "none" (the matched control: same network, NO alignment penalty)
QUICK_TEST = False                 # True = 2 splits, 2 epochs, a small weight sweep. False = the real run.

SOURCE_METHOD = "source_only"
TRAIN_SEED = 42

OUT_DIR = Path("outputs")
MANIFEST = OUT_DIR / "master_manifest.csv"
SEEDS = list(range(42, 52))       # the 10 Setting B splits
DEV_SEED = 42                     # the split used to choose the weight

# Our choices (the papers do not fix these for our setting)
BOTTLENECK = 256
BATCH_SIZE = 128                  # source images per step (and the same number of field images)
EPOCHS = 20
PATIENCE = 5                      # stop if lab-validation accuracy has not improved for this many epochs
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.01
TOLERANCE = 0.01                  # the chosen weight may cost at most this much lab-validation accuracy
SCALE_FLOOR = 0.5                 # ... and must keep the bottleneck features at least this fraction of their no-alignment size
EPS = 1e-6

# Weights to try (the same number of values for both methods). The two losses live on very different scales.
WEIGHT_GRID = {"coral": [10, 100, 1000, 10000, 100000], "jmmd": [0.1, 0.3, 1, 3, 10], "none": []}
# -----------------------------------------------------------------------------

METHOD = {"coral": "deep_coral", "jmmd": "jmmd", "none": "bottleneck_only"}[ALIGNMENT]
weight_grid = WEIGHT_GRID[ALIGNMENT]
if QUICK_TEST:
    EPOCHS, weight_grid = 2, weight_grid[1:3]
seeds_to_run = SEEDS[:2] if QUICK_TEST else SEEDS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Method: {METHOD} | device: {device} | splits: {seeds_to_run} | epochs: {EPOCHS} | weights to try: {weight_grid}")
if QUICK_TEST:
    print("*** QUICK TEST: 2 splits, 2 epochs, 2 weights. Set QUICK_TEST = False for the real run. ***")

Method: jmmd | device: cuda | splits: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51] | epochs: 20 | weights to try: [0.1, 0.3, 1, 3, 10]


## Step 2: Load the features

Same data as notebook 4c: PlantVillage training features (from 02b), PlantVillage validation features and the PlantDoc
features (from 02). All are **standardised with the PlantVillage training statistics**, which does not adapt anything to the
field.

In [16]:
source = np.load(OUT_DIR / f"outputs_{SOURCE_METHOD}_seed{TRAIN_SEED}.npz", allow_pickle=True)
classes = list(source["classes"])
n_classes = len(classes)

train = np.load(OUT_DIR / f"features_pv_train_{SOURCE_METHOD}_seed{TRAIN_SEED}.npz", allow_pickle=True)
train_features, train_labels = train["features"].astype(np.float32), train["labels"]

val_features, val_labels = source["pv_val_features"].astype(np.float32), source["pv_val_labels"]
known_features = source["pd_known_features"].astype(np.float32)
unknown_features = source["pd_unknown_features"].astype(np.float32)
field_labels = source["pd_known_labels"]
known_paths = source["pd_known_paths"]

mean, std = train_features.mean(axis=0), train_features.std(axis=0)
to_tensor = lambda x: torch.from_numpy((x - mean) / (std + EPS)).to(device)

src_x, val_x = to_tensor(train_features), to_tensor(val_features)
known_x, unknown_x = to_tensor(known_features), to_tensor(unknown_features)
src_y = torch.from_numpy(train_labels).long().to(device)
val_y = torch.from_numpy(val_labels).long().to(device)

manifest = pd.read_csv(MANIFEST, keep_default_na=False)
split_columns = [f"B_seed{s}" for s in SEEDS]
known_rows = manifest[(manifest["dataset"] == "pd") & (manifest["role"] == "pd_known")].set_index("path")
split_info = known_rows.loc[known_paths, split_columns].reset_index(drop=True)

print("source train:", tuple(src_x.shape), "| source validation:", tuple(val_x.shape),
      "| field known:", tuple(known_x.shape), "| field unknown:", tuple(unknown_x.shape))

source train: (18601, 2048) | source validation: (3986, 2048) | field known: (2018, 2048) | field unknown: (466, 2048)


## Step 3: The two alignment losses

Both take a batch of **source** features and a batch of **target** (field) features and return **one number**: 0 when
the two batches look alike, larger when they differ.

### Deep CORAL
1. Compute each batch's **covariance matrix** (how every pair of features varies together).
2. Take the difference of the two matrices, square every entry, and add them up.
3. Divide by 4 x d x d (d = number of features), so the size does not depend on d.

In [17]:
def coral_loss(source_feat, target_feat):
    d = source_feat.shape[1]

    def covariance(x):
        x = x - x.mean(dim=0, keepdim=True)            # remove the average: covariance ignores where the data sits
        return x.t() @ x / (len(x) - 1)

    difference = covariance(source_feat) - covariance(target_feat)
    return (difference ** 2).sum() / (4 * d * d)

**Check it on toy data.** Two clouds with the **same shape but different positions** should score about 0, because CORAL
ignores averages. A cloud and a **stretched** version of it should score clearly above 0.

In [18]:
torch.manual_seed(0)
a = torch.randn(500, 8)
print("same shape, shifted position:", round(float(coral_loss(a, a + 5.0)), 6), "(expect ~0: averages are ignored)")
print("same cloud, stretched 2x:    ", round(float(coral_loss(a, a * 2.0)), 6), "(expect clearly above 0)")

same shape, shifted position: 0.0 (expect ~0: averages are ignored)
same cloud, stretched 2x:     0.278833 (expect clearly above 0)


### JMMD (joint maximum mean discrepancy)
The idea of **MMD**: measure how similar two sets of points are with a *kernel*, a function that is near 1 for points
that are close and near 0 for points that are far apart. Then

**MMD = (average similarity inside source) + (average similarity inside target) - 2 x (average similarity between source and target)**

If the two sets overlap well, all three averages are similar and MMD is near 0. If they sit apart, the similarity
*between* them is low and MMD is large. Unlike CORAL it notices differences in **position** as well as shape, and in the full
distribution, not only in covariance.

**"Joint"** means we apply this to the bottleneck features **and** the predicted class probabilities **at the same time**, by
**multiplying their kernels**: two images count as similar only if they match in *both*.

We use several kernel widths at once, so that we do not have to tune one. (The width is set from the data: the average
squared distance between points, then 5 widths spaced by factors of 2.)

In [19]:
def kernel_matrix(x, kernel_mul=2.0, kernel_num=5):
    # Similarity between every pair of rows in x, using 5 Gaussian kernels of different widths added together
    squared = (x * x).sum(dim=1, keepdim=True)
    distance2 = (squared + squared.t() - 2 * x @ x.t()).clamp(min=0)         # squared distances, no sqrt (safe to differentiate)
    n = distance2.shape[0]
    width = distance2.detach().sum() / (n * n - n)                           # average squared distance between different points
    width = width / (kernel_mul ** (kernel_num // 2))
    return sum(torch.exp(-distance2 / (width * kernel_mul ** i)) for i in range(kernel_num))


def jmmd_loss(source_layers, target_layers):
    # source_layers / target_layers: lists such as [bottleneck features, class probabilities]
    n = len(source_layers[0])
    joint = None
    for s, t in zip(source_layers, target_layers):
        k = kernel_matrix(torch.cat([s, t]))
        joint = k if joint is None else joint * k                            # multiply the kernels of the different layers
    k_ss, k_tt, k_st = joint[:n, :n], joint[n:, n:], joint[:n, n:]
    return k_ss.mean() + k_tt.mean() - 2 * k_st.mean()

**Check it on toy data.** Two samples from the **same** distribution should score near 0. A **shifted** sample should score
clearly higher, and a larger shift should score higher still.

In [20]:
torch.manual_seed(0)
a, b = torch.randn(300, 8), torch.randn(300, 8)
print("same distribution:   ", round(float(jmmd_loss([a], [b])), 4))
print("shifted by 1:        ", round(float(jmmd_loss([a], [b + 1.0])), 4))
print("shifted by 3:        ", round(float(jmmd_loss([a], [b + 3.0])), 4), "(expect: increasing)")

same distribution:    0.0255
shifted by 1:         0.9838
shifted by 3:         4.164 (expect: increasing)


## Step 4: The model and the training function

The model is the same small classifier as in notebook 4c, **without** the discriminator: a bottleneck (2048 to 256 numbers
with a ReLU) followed by a linear classifier (256 to 21 classes).

The training function runs the loop and combines the losses. The alignment weight is **ramped up gradually** from 0
(the same smooth schedule as in 4c), so the network first learns to classify and only later is pulled toward the field.
Stopping uses lab-validation accuracy, as before. It also returns the final alignment loss and the lab-validation
accuracy, which Step 5 needs.

In [21]:
class AlignNet(nn.Module):
    def __init__(self, in_dim, n_classes, bottleneck=BOTTLENECK):
        super().__init__()
        self.bottleneck = nn.Sequential(nn.Linear(in_dim, bottleneck), nn.ReLU())
        self.classifier = nn.Linear(bottleneck, n_classes)

    def forward(self, x):
        feat = self.bottleneck(x)
        return self.classifier(feat), feat


def alignment_loss(logits_s, feat_s, logits_t, feat_t):
    if ALIGNMENT == "none":
        return feat_s.sum() * 0.0                       # the control: no penalty at all
    if ALIGNMENT == "coral":
        return coral_loss(feat_s, feat_t)
    return jmmd_loss([feat_s, F.softmax(logits_s, dim=1)], [feat_t, F.softmax(logits_t, dim=1)])

In [22]:
def train_aligned(target_x, seed, weight, verbose=False):
    torch.manual_seed(seed)
    net = AlignNet(src_x.shape[1], n_classes).to(device)
    optimizer = torch.optim.AdamW(net.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    steps_per_epoch = len(src_x) // BATCH_SIZE
    total_steps = EPOCHS * steps_per_epoch
    best_acc, best_state, bad_epochs, last_align = -1.0, None, 0, float("nan")

    for epoch in range(EPOCHS):
        net.train()
        order = torch.randperm(len(src_x), device=device)

        for step in range(steps_per_epoch):
            progress = (epoch * steps_per_epoch + step) / total_steps
            ramp = 2.0 / (1.0 + np.exp(-10.0 * progress)) - 1.0           # 0 at the start, close to 1 at the end

            idx = order[step * BATCH_SIZE:(step + 1) * BATCH_SIZE]
            xs, ys = src_x[idx], src_y[idx]
            xt = target_x[torch.randint(len(target_x), (BATCH_SIZE,), device=device)]    # unlabelled field batch

            logits_s, feat_s = net(xs)
            logits_t, feat_t = net(xt)
            classification_loss = F.cross_entropy(logits_s, ys)
            align = alignment_loss(logits_s, feat_s, logits_t, feat_t)

            optimizer.zero_grad()
            (classification_loss + weight * ramp * align).backward()
            optimizer.step()
            last_align = float(align)

        net.eval()
        with torch.no_grad():
            val_acc = float((net(val_x)[0].argmax(dim=1) == val_y).float().mean())
        if verbose:
            print(f"epoch {epoch + 1:2d} | class loss {classification_loss.item():.3f} | align loss {last_align:.4f} | "
                  f"lab validation accuracy {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc, bad_epochs = val_acc, 0
            best_state = {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break

    net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        feature_size = float(net(val_x)[1].norm(dim=1).mean())          # average length of the bottleneck feature vector
    return net, best_acc, last_align, feature_size


@torch.no_grad()
def logits_of(net, x):
    return net(x)[0].cpu().numpy().astype(np.float32)

## Step 5: Choose the weight

On the development split we train once with **weight 0** (no alignment, the reference) and then once per candidate weight.
For each we record the lab-validation accuracy. **Rule: pick the largest weight whose lab-validation accuracy is within
`TOLERANCE` (1 point) of the weight-0 run AND whose `feature size` is at least `SCALE_FLOOR` (half) of the weight-0 run.** Intuition: align as strongly as you can without damaging what the model
learned on the lab images and without the trivial trick of making all the features small.

We also show the **average prediction entropy on the field calibration images** (how undecided the model is on them) for
information. We do not use it to choose, and we never look at field labels here.

In [23]:
dev_cal = (split_info[f"B_seed{DEV_SEED}"] == "cal").to_numpy()
dev_target = known_x[torch.from_numpy(dev_cal).to(device)]
print(f"Development split {DEV_SEED}: {len(dev_target)} unlabelled field images")

rows = []
for weight in ([] if ALIGNMENT == "none" else [0] + list(weight_grid)):
    start = time.time()
    net, val_acc, align_value, feature_size = train_aligned(dev_target, DEV_SEED, weight)
    with torch.no_grad():
        probs = F.softmax(net(dev_target)[0], dim=1)
        entropy = float(-(probs * torch.log(probs.clamp_min(1e-12))).sum(dim=1).mean())
    rows.append({"weight": weight, "lab validation accuracy": round(val_acc, 4),
                 "final alignment loss": round(align_value, 5), "feature size": round(feature_size, 3),
                 "field entropy (info only)": round(entropy, 3),
                 "seconds": round(time.time() - start)})
    print(rows[-1])

sweep = pd.DataFrame(rows).set_index("weight") if rows else None
sweep

Development split 42: 202 unlabelled field images


{'weight': 0, 'lab validation accuracy': 0.9975, 'final alignment loss': 2.8224, 'feature size': 51.352, 'field entropy (info only)': 0.354, 'seconds': 10}
{'weight': 0.1, 'lab validation accuracy': 0.9957, 'final alignment loss': 0.33582, 'feature size': 48.38, 'field entropy (info only)': 0.249, 'seconds': 9}
{'weight': 0.3, 'lab validation accuracy': 0.9957, 'final alignment loss': 0.37851, 'feature size': 33.997, 'field entropy (info only)': 0.104, 'seconds': 12}
{'weight': 1, 'lab validation accuracy': 0.9945, 'final alignment loss': 0.31994, 'feature size': 41.979, 'field entropy (info only)': 0.176, 'seconds': 8}
{'weight': 3, 'lab validation accuracy': 0.9925, 'final alignment loss': 0.30848, 'feature size': 41.401, 'field entropy (info only)': 0.17, 'seconds': 8}
{'weight': 10, 'lab validation accuracy': 0.9872, 'final alignment loss': 0.25862, 'feature size': 30.556, 'field entropy (info only)': 0.129, 'seconds': 10}


,lab validation accuracy,final alignment loss,feature size,field entropy (info only),seconds
weight,,,,,
0.0,0.9975,2.82240,51.352,0.354,10
0.1,0.9957,0.33582,48.380,0.249,9
0.3,0.9957,0.37851,33.997,0.104,12
1.0,0.9945,0.31994,41.979,0.176,8
3.0,0.9925,0.30848,41.401,0.170,8
10.0,0.9872,0.25862,30.556,0.129,10


In [24]:
if ALIGNMENT == "none":
    chosen_weight = 0
    print("Control run: no alignment, so there is no weight to choose.")
else:
    reference = sweep.loc[0, "lab validation accuracy"]
    reference_size = sweep.loc[0, "feature size"]

    keeps_accuracy = [w for w in weight_grid if sweep.loc[w, "lab validation accuracy"] >= reference - TOLERANCE]
    keeps_size = [w for w in weight_grid if sweep.loc[w, "feature size"] >= SCALE_FLOOR * reference_size]
    allowed = [w for w in weight_grid if w in keeps_accuracy and w in keeps_size]
    chosen_weight = max(allowed) if allowed else min(weight_grid)

    print(f"Lab validation accuracy with no alignment: {reference:.4f} | feature size with no alignment: {reference_size:.3f}")
    print(f"Weights that keep the lab accuracy (within {TOLERANCE}): {keeps_accuracy}")
    print(f"Weights that keep the features at least {SCALE_FLOOR:.0%} of their size:  {keeps_size}")
    print(f"Chosen weight: {chosen_weight}", "" if allowed else "(none qualified, so we take the smallest)")
    if chosen_weight == max(weight_grid):
        print("*** The chosen weight is the TOP of the grid: extend WEIGHT_GRID upwards and re-run Step 5. ***")

Lab validation accuracy with no alignment: 0.9975 | feature size with no alignment: 51.352
Weights that keep the lab accuracy (within 0.01): [0.1, 0.3, 1, 3]
Weights that keep the features at least 50% of their size:  [0.1, 0.3, 1, 3, 10]
Chosen weight: 3 


**Questions to think about:**
* Does the alignment loss go **down** as the weight goes up? If it doesn't, the weight is too small to matter.
* At what weight does the lab accuracy start to suffer? What does that say about the trade-off between fitting the lab
  and matching the field?
* The rule picks the strongest weight that is "safe" for the lab. Why can't we use field accuracy to choose instead?

## Step 6: All splits, then save

For each split, we train with the chosen weight, using that split's calibration images as the unlabelled field batch, and
keep the logits for all known and unknown PlantDoc images. (Training uses no field labels.)

In [25]:
def accuracy(logits, mask):
    return float((logits[mask].argmax(axis=1) == field_labels[mask]).mean())


save_dict = {"classes": np.array(classes), "align_weight": np.array(chosen_weight)}
for key in ["epochs_run", "debug_mode"]:
    if key in source.files:
        save_dict[key] = source[key]
for g in ["pv_val", "pv_test"]:
    save_dict[f"{g}_logits"] = source[f"{g}_logits"]
for g in ["pv_val", "pv_test", "pd_known", "pd_unknown"]:
    save_dict[f"{g}_labels"] = source[f"{g}_labels"]
    save_dict[f"{g}_paths"] = source[f"{g}_paths"]

for seed in seeds_to_run:
    start = time.time()
    is_cal = (split_info[f"B_seed{seed}"] == "cal").to_numpy()
    net, *_ = train_aligned(known_x[torch.from_numpy(is_cal).to(device)], seed, chosen_weight)

    save_dict[f"pd_known_logits_B_seed{seed}"] = logits_of(net, known_x)
    save_dict[f"pd_unknown_logits_B_seed{seed}"] = logits_of(net, unknown_x)
    print(f"split {seed}: {time.time() - start:.0f} s | test accuracy "
          f"{accuracy(save_dict[f'pd_known_logits_B_seed{seed}'], ~is_cal):.4f}")

suffix = "_QUICK" if QUICK_TEST else ""
output_file = OUT_DIR / f"outputs_{METHOD}_seed{TRAIN_SEED}{suffix}.npz"
np.savez_compressed(output_file, **save_dict)
print("Saved", output_file)

split 42: 8 s | test accuracy 0.2539
split 43: 9 s | test accuracy 0.2500
split 44: 14 s | test accuracy 0.2494
split 45: 16 s | test accuracy 0.2296
split 46: 13 s | test accuracy 0.2595
split 47: 15 s | test accuracy 0.2434
split 48: 14 s | test accuracy 0.2672
split 49: 16 s | test accuracy 0.2329
split 50: 17 s | test accuracy 0.2472
split 51: 15 s | test accuracy 0.2687
Saved outputs\outputs_jmmd_seed42.npz


## Step 7: Evaluate with notebook 3b

1. Run this notebook with `ALIGNMENT = "coral"` and `QUICK_TEST = True` to check that it works. Then set
   `QUICK_TEST = False`, restart the kernel and run it again. The real file is `outputs_deep_coral_seed0.npz`.
2. Change `ALIGNMENT = "jmmd"` and repeat both runs. The real file is `outputs_jmmd_seed0.npz`.
3. Run it a third time with `ALIGNMENT = "none"`. This is the matched control (`outputs_bottleneck_only_seed0.npz`).
4. In notebook 3b add `"deep_coral"`, `"jmmd"` and `"bottleneck_only"` to `METHODS` and run it. Tables A, B and C will then include them.

**What to look for (no reference values exist for these methods).** Compare them with `source_only`, `linear_head`, `adabn`,
`moment_matching` and `dann`:
* Does a **richer** alignment (CORAL, then JMMD) beat the simple baselines in accuracy, Risk@80 and the **raw** gap in
  Table C?
* Does aligning help the model on **unknown** diseases (AUROC, AUPR), or hurt it, as the baselines did?
* Is anything larger than the split-to-split spread (the ± values)?

This is the question of our proposal: do deeper alignment methods share the fate of the lightweight ones?

### Next
**Notebook 6: JDOT and DeepJDOT** (optimal transport).